# Doublet Analysis with fourth sequencing run (Doublets_4_2024). Use Hugo Symbols, not Ensembl ID.

Publication source copy for the Doublets 2 and 4 sequencing analyses. Outputs and execution counts are cleared. See `../VALIDATION_REPORT.md` for the separate execution results and remaining limitations.

Run from the `analysis/` directory after supplying the bundled inputs. Final figure sections explicitly load preserved manuscript checkpoints; they are distinct from fresh clustering.

In [ ]:
from pathlib import Path
Path("../results/Doublets_4_Step1_PreProcessing_halveSingletReads_with_DoubletScores_only_HugoGeneSymbols").mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items #pandas version incompatibility fix. Reference: https://github.com/YosefLab/Compass/issues/92
import numpy as np
import scanpy as sc
import scipy as sp
from anndata import read_h5ad
from anndata import read_text
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import rc_context

import anndata
import seaborn as sns
import regex as re
import os

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, dpi_save=300)
sc.settings.n_jobs = 30
pd.options.display.max_columns = None
pd.options.display.max_rows = None

sc.logging.print_header()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Convert Ensembl ID to Hugo Gene Symbols in gene count matrix

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
df = pd.read_csv(path_data+'genetables.2p/Doublets4_RSM_gRDS_and_stats_table_clean.tsv.gz', sep='\t')
df.head()

# #get gene symbols
df['01_gene_id'] = [i.split('_')[-1] for i in df['01_gene_id']]
df.head()

# Create a boolean mask to identify duplicates in gene symbol
gene_symbol_mask = df['01_gene_id'].duplicated(keep='first')

# Use the mask to filter out the duplicate rows
df_no_duplicates = df[~gene_symbol_mask]
df_no_duplicates.head()

In [ ]:
#save as compressed file
df_no_duplicates.to_csv(path_data+'genetables.2p/Doublets4_RSM_gRDS_and_stats_table_clean_GeneSymbol_NoDuplicates.tsv.gz', compression='gzip', sep='\t', index=False)

# Import Doublet data from tsv.gz

Only need to do this once. Use **h5ad** file after this step.

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_text(path_data+'genetables.2p/Doublets4_RSM_gRDS_and_stats_table_clean_GeneSymbol_NoDuplicates.tsv.gz')
adata

In [ ]:
#Extract metadata from cell name
adata = adata.transpose()
adata.obs['SequencingRun'] = [i.split('_')[0] for i in adata.obs_names] 
adata.obs['SampleID'] = [i.split('_')[1] for i in adata.obs_names] 
adata.obs['SampleType'] = [i.split('_')[2].split('-')[0] for i in adata.obs_names]
adata.obs['Stimulation'] = [i.split('_')[2].split('-')[1] for i in adata.obs_names]
adata.obs['age'] = [i.split('_')[3] for i in adata.obs_names]
adata.obs['sex'] = [i.split('_')[4] for i in adata.obs_names]
adata.obs['sortedtype'] = [i.split('_')[5][0:-2] for i in adata.obs_names] #Rename this to whatever you called it last time. It's "Doublet-Real", "Doublet-Artificial", or "Singlet"
adata.obs['score'] = [i.split('_')[5][-1] for i in adata.obs_names]
adata.obs['sortedpop'] = [i.split('_')[6] for i in adata.obs_names]
adata.obs['96plate'] = [i.split('_')[7] for i in adata.obs_names]
adata.obs['96plateloc'] = [i.split('_')[8] for i in adata.obs_names]
adata.obs['384plate'] = [i.split('_')[9] for i in adata.obs_names]
adata.obs['384plateloc'] = [i.split('_')[10] for i in adata.obs_names]
adata.obs['barcode'] = [i.split('_')[11] for i in adata.obs_names]
adata.obs['cell'] = adata.obs_names

# Write h5ad file

Do not overwrite!

In [ ]:
# #save tcr data without preprocessing
adata.write_h5ad(path_data+'Doublets4_7680cells_GeneSymbols_nopreprocessing.h5ad')

# Quick load adata from h5ad with TCR and BCR info as bool values

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = read_h5ad(path_data+'Doublets4_7680cells_GeneSymbols_nopreprocessing.h5ad')

adata.obs.head()

# Quality Control

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
# Remove ERCC reads
adata.obs['n_countswercc'] = np.sum(adata.X, axis=1)
ercc_gene_mask = [gene.split('_')[0].startswith('ERCC-') for gene in adata.var_names]
adata.obs['ercc_frac'] = adata.X[:, ercc_gene_mask].sum(1)/adata.obs['n_countswercc']

ercc_genes = [name for name in adata.var_names if name.startswith('ERCC-')]
adata = adata[:,~adata.var_names.isin(ercc_genes)]
adata

In [ ]:
adata.obs['n_counts'] = np.sum(adata.X, axis=1)
adata

In [ ]:
# Quality control - calculate QC covariates
adata.obs['n_counts'] = adata.X.sum(1)
adata.obs['n_genes'] = (adata.X > 0).sum(1)
# Mitochondrial genes

mt_gene_mask = [gene.startswith('MT-') for gene in adata.var_names]

adata.obs['mt_frac'] = adata.X[:, mt_gene_mask].sum(1)/adata.obs['n_counts']

In [ ]:
sc.pl.violin(adata, ['n_genes','mt_frac','ercc_frac'],groupby='sortedpop',jitter=0.4,stripplot=True,multi_panel=True,rotation=90)

In [ ]:
sc.pl.violin(adata, ['n_genes', 'n_counts', 'mt_frac'], jitter=0.4, multi_panel=True)
sc.pl.scatter(adata, x='n_counts', y='mt_frac')
sc.pl.scatter(adata, x='n_counts', y='n_genes')
sc.pl.scatter(adata, x='n_countswercc', y='ercc_frac')
sc.pl.scatter(adata, x='n_genes', y='ercc_frac')

# Filter out bad cells

In [ ]:
#filter cells for gene counts
print(len(adata))

sc.pp.filter_cells(adata, min_genes=500)
sc.pp.filter_genes(adata, min_cells=3)
adata.shape

In [ ]:
#filter cells for mtrna and ercc
adata = adata[adata.obs.mt_frac < 0.5, :]
adata = adata[adata.obs.ercc_frac < 0.2, :] 
adata.shape

# Pre-Processing - halve singlet normalized counts

In [ ]:
Counter(adata.obs['sortedtype'])

In [ ]:
adata_singlet = adata[adata.obs['sortedtype']=='Sing']
adata_singlet.shape
adata_doublet = adata[(adata.obs['sortedtype']=='ArtDoub') | (adata.obs['sortedtype']=='RealDoub')]
adata_doublet.shape

In [ ]:
sc.pp.normalize_total(adata_singlet, target_sum=1e4)
sc.pp.normalize_total(adata_doublet, target_sum=1e4)

In [ ]:
# #double counts for doublets

# adata_doublet.X

#halve counts for singlets
adata_singlet.X = 0.5*adata_singlet.X
adata_singlet.X

In [ ]:
adata = adata_doublet.concatenate(adata_singlet,index_unique=None)
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata

In [ ]:
adata.uns['log1p']["base"] = None
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.var.head()

# Scale

In [ ]:
sc.pp.scale(adata, max_value=10)

# Save processed data

Do not overwrite!

In [ ]:
adata

In [ ]:
#save file
adata.write_h5ad(path_data+'Doublets4_7600cells_GeneSymbols_preprocessed_2xDoubletCounts.h5ad')

# read data

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad(path_data+'Doublets4_7600cells_GeneSymbols_preprocessed_2xDoubletCounts.h5ad')

#the line below fixes an error that results when trying to run sc.tl.rank_genes_groups(adata,'leiden',method='wilcoxon')
#error is: KeyError: 'base'
adata.uns['log1p']["base"] = None
adata.obs.head()

# QC Check

In [ ]:
sc.pl.violin(adata, keys=['n_counts', 'n_genes', 'mt_frac', 'ercc_frac'], groupby='sortedtype', rotation=90, use_raw=True)
sc.pl.violin(adata, keys=['n_counts', 'n_genes', 'mt_frac', 'ercc_frac'], groupby='sortedpop', rotation=90, use_raw=True)

# Average values

In [ ]:
adata.obs['n_genes'].mean()